# C05: Manejo de Archivos y Excepciones

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Crear y navegar rutas de archivos usando `pathlib` de forma segura y portable.
2. Leer y escribir archivos de texto y binarios usando context managers (`with`).
3. Serializar y deserializar datos con los módulos `json` y `csv`.
4. Manejar errores de forma estructurada con `try / except / else / finally`.
5. Crear excepciones propias y usar `assert` para validaciones.

## Analogía: el avión y el archivador

Imagina que estás a bordo de un avión:

- **Excepciones = señales de emergencia.** Si algo sale mal (motor falla, turbulencia fuerte), el piloto recibe una señal y ejecuta un protocolo: intenta corregir, reporta a la torre de control y, si nada funciona, aterriza de emergencia. En Python, `try` es *intentar la maniobra*, `except` es *actuar ante la señal*, `else` es *confirmar que todo salió bien* y `finally` es *asegurar que los chalecos estén colocados sin importar qué*.
- **Archivos = archivadores físicos.** Cada archivo es un cajón dentro de una oficina. La **ruta** (`path`) es la dirección exacta del cajón: desde la puerta de la oficina (raíz) hasta el cajón específico. `pathlib` es como un asistente que conoce la oficina y te guía sin equivocarte.
- **Modos de apertura** = formas de interactuar con el cajón: puedes *leer* su contenido (`r`), *escribir* algo nuevo (`w`), *agregar* hojas al final (`a`), o crear uno nuevo que no existía (`x`).

## 1. Rutas con `pathlib`

`pathlib` (disponible desde Python 3.4) representa las rutas del sistema como objetos, en lugar de strings crudos. Esto evita problemas con `/` vs `\` entre sistemas operativos.

### 1.1 Crear rutas

In [ ]:
from pathlib import Path

# Ruta absoluta (desde la raíz del disco)
ruta_abs = Path(r"C:\Users\luisj\Documents")
print(f"Ruta absoluta: {ruta_abs}")

# Ruta relativa (desde el directorio actual)
ruta_rel = Path("datos/ventas.csv")
print(f"Ruta relativa: {ruta_rel}")

# Usando / operador para combinar (portable entre SO)
ruta_combinada = Path("datos") / "ventas" / "enero.csv"
print(f"Combinada:     {ruta_combinada}")

### 1.2 Propiedades de una ruta

In [ ]:
p = Path("C:/Users/luisj/datos/reporte_final.xlsx")

print(f"Nombre completo: {p.name}")      # reporte_final.xlsx
print(f"Sin extensión:   {p.stem}")      # reporte_final
print(f"Extensión:       {p.suffix}")    # .xlsx
print(f"Directorio:      {p.parent}")    # C:/Users/luisj/datos
print(f"Raíz:            {p.root}")      # C:\
print(f"¿Es absoluto?    {p.is_absolute()}")

### 1.3 Operaciones comunes

In [ ]:
from pathlib import Path

# Crear un directorio de ejemplo
carpeta_ejemplo = Path("ejemplo_archivos")
carpeta_ejemplo.mkdir(exist_ok=True)
print(f"¿Existe la carpeta? {carpeta_ejemplo.exists()}")
print(f"¿Es un directorio?  {carpeta_ejemplo.is_dir()}")

# Escribir y leer un archivo de texto
archivo = carpeta_ejemplo / "saludo.txt"
archivo.write_text("Hola desde pathlib!", encoding="utf-8")
print(f"Contenido: {archivo.read_text(encoding='utf-8')}")
print(f"¿Es archivo? {archivo.is_file()}")

# Listar archivos con un patrón
print(f"Archivos .txt: {list(carpeta_ejemplo.glob('*.txt'))}")

In [ ]:
# Limpiar archivos de ejemplo
from pathlib import Path

carpeta = Path("ejemplo_archivos")
for archivo in carpeta.iterdir():
    archivo.unlink()          # eliminar archivo
carpeta.rmdir()               # eliminar directorio vacío
print("Archivos de ejemplo eliminados.")

## 2. Context managers (`with`)

El bloque `with` garantiza que el archivo se **cierre** automáticamente al salir, aunque ocurra un error. Es la forma segura y recomendada de trabajar con archivos.

### 2.1 ¿Por qué usar `with`?

| Sin `with` | Con `with` |
|---|---|
| `f = open(...)` al inicio | Se abre y se cierra solo |
| Debes recordar `f.close()` | No olvidas cerrar nunca |
| Si hay excepción, el archivo queda abierto | Se cierra incluso con errores |

### 2.2 Modos de apertura

In [ ]:
from pathlib import Path

carpeta = Path("ejemplo_archivos")
carpeta.mkdir(exist_ok=True)
ruta = carpeta / "datos.txt"

# 'w' — write: sobrescribe el archivo completo
with open(ruta, "w", encoding="utf-8") as f:
    f.write("Primera línea\n")
    f.write("Segunda línea\n")

# 'a' — append: agrega al final sin borrar
with open(ruta, "a", encoding="utf-8") as f:
    f.write("Tercera línea (agregada)\n")

# 'r' — read: leer el contenido
with open(ruta, "r", encoding="utf-8") as f:
    contenido = f.read()
print(contenido)

In [ ]:
# 'x' — exclusive creation: falla si el archivo ya existe
ruta_nueva = carpeta / "nuevo.txt"
try:
    with open(ruta_nueva, "x", encoding="utf-8") as f:
        f.write("Archivo creado por primera vez\n")
    print("Archivo creado exitosamente.")
except FileExistsError:
    print("El archivo ya existía. No se sobrescribe.")

# Intentar de nuevo → dispara el error
try:
    with open(ruta_nueva, "x", encoding="utf-8") as f:
        f.write("Esto no debería escribirse\n")
except FileExistsError as e:
    print(f"Error: {e}")

In [ ]:
# Archivos binarios: modo 'rb' y 'wb'
ruta_bin = carpeta / "datos.bin"

datos_bytes = b"\x00\x01\x02\x03\xff"
with open(ruta_bin, "wb") as f:
    f.write(datos_bytes)

with open(ruta_bin, "rb") as f:
    leido = f.read()
print(f"Leído (binario): {leido}")
print(f"Longitud: {len(leido)} bytes")

In [ ]:
# Lectura línea por línea (eficiente para archivos grandes)
with open(ruta, "r", encoding="utf-8") as f:
    for i, linea in enumerate(f, start=1):
        print(f"Línea {i}: {linea.strip()}")

## 3. JSON

JSON (JavaScript Object Notation) es el formato de intercambio más popular para datos estructurados. Python lo maneja con el módulo `json`.

In [ ]:
import json
from pathlib import Path

carpeta = Path("ejemplo_archivos")
ruta_json = carpeta / "empleados.json"

# Datos anidados (dicts y listas)
empleados = [
    {
        "nombre": "Ana García",
        "edad": 32,
        "departamento": "Ingeniería",
        "habilidades": ["Python", "SQL", "Docker"]
    },
    {
        "nombre": "Carlos López",
        "edad": 28,
        "departamento": "Datos",
        "habilidades": ["Spark", "Pandas", "Airflow"]
    },
    {
        "nombre": "María Ruiz",
        "edad": 41,
        "departamento": "Ingeniería",
        "habilidades": ["Java", "Kubernetes", "AWS"]
    }
]

# json.dump — escribe directamente al archivo
with open(ruta_json, "w", encoding="utf-8") as f:
    json.dump(empleados, f, indent=2, ensure_ascii=False)

print("Archivo JSON creado.")
print()

# json.load — lee y convierte a Python
with open(ruta_json, "r", encoding="utf-8") as f:
    datos = json.load(f)

for emp in datos:
    print(f"{emp['nombre']} — {emp['departamento']}")

In [ ]:
# json.dumps — convierte a string (útil para APIs, logs, depuración)
empleado = datos[0]

# Sin formato
linea = json.dumps(empleado, ensure_ascii=False)
print(f"Sin formato: {linea[:80]}...")

# Con indentación y separadores personalizados
bonito = json.dumps(
    empleado,
    indent=4,
    ensure_ascii=False,
    separators=(", ", ": ")
)
print()
print(bonito)

### Diagrama ASCII — Flujo `try / except / else / finally`

```
                    ┌────────────────────┐
                    │  Código que puede   │
                    │  fallar (bloque     │
                    │       try)          │
                    └────────┬───────────┘
                             │
                ┌────────────┴────────────┐
                │                         │
           ¿Falló?                   ¿Éxito?
           (except)                 (else)
                │                         │
        ┌───────┴───────┐         ┌───────┴───────┐
        │ Ejecuta el     │         │ Ejecuta el     │
        │ bloque except  │         │ bloque else    │
        │ correspondiente│         │ (si existe)    │
        └───────┬───────┘         └───────┬───────┘
                │                         │
                └────────────┬────────────┘
                             │
                    ┌────────┴───────────┐
                    │   finally SIEMPRE  │
                    │   se ejecuta       │
                    └────────────────────┘
```

In [ ]:
import json
from pathlib import Path

carpeta = Path("ejemplo_archivos")
ruta_config = carpeta / "config.json"

# Ejemplo completo: leer JSON con manejo de errores
try:
    with open(ruta_config, "r", encoding="utf-8") as f:
        config = json.load(f)
except FileNotFoundError:
    print("Archivo no encontrado. Se usa configuración por defecto.")
    config = {"version": "1.0", "debug": False}
except json.JSONDecodeError as e:
    print(f"JSON inválido: {e}")
    config = {}
else:
    print("Configuración cargada correctamente.")
finally:
    print(f"Config actual: {config}")

## 4. CSV

CSV (Comma-Separated Values) es el formato tabular más común. El módulo `csv` lo maneja sin necesidad de instalar librerías externas.

In [ ]:
import csv
from pathlib import Path

carpeta = Path("ejemplo_archivos")
ruta_ventas = carpeta / "ventas.csv"

# Datos de ejemplo
ventas = [
    ["producto", "cantidad", "precio_unitario"],
    ["Laptop", 15, 899.99],
    ["Mouse", 120, 24.50],
    ["Teclado", 85, 49.99],
    ["Monitor", 30, 349.00],
    ["Webcam", 55, 79.95]
]

# csv.writer — escribe filas como listas
with open(ruta_ventas, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(ventas)

print("Archivo CSV creado.")

In [ ]:
# csv.reader — lee cada fila como lista
print("=== csv.reader ===")
with open(ruta_ventas, "r", encoding="utf-8") as f:
    reader = csv.reader(f)
    for fila in reader:
        print(fila)

print()

# csv.DictReader — lee cada fila como diccionario (usando la cabecera)
print("=== csv.DictReader ===")
with open(ruta_ventas, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for fila in reader:
        total = int(fila["cantidad"]) * float(fila["precio_unitario"])
        print(f"{fila['producto']:10s} — Total: ${total:,.2f}")

In [ ]:
import csv
from pathlib import Path

carpeta = Path("ejemplo_archivos")
ruta_reporte = carpeta / "reporte_ventas.csv"

# csv.DictWriter — escribe desde diccionarios
datos_reporte = [
    {"producto": "Laptop", "vendidas": 15, "region": "Norte"},
    {"producto": "Mouse", "vendidas": 120, "region": "Sur"},
    {"producto": "Teclado", "vendidas": 85, "region": "Este"},
]

with open(ruta_reporte, "w", newline="", encoding="utf-8") as f:
    campos = ["producto", "vendidas", "region"]
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()       # escribe la fila de cabecera
    writer.writerows(datos_reporte)

print("Reporte CSV creado.")

# Leerlo de vuelta para verificar
with open(ruta_reporte, "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for fila in reader:
        print(dict(fila))

## 5. Excepciones

Las excepciones son la forma que Python tiene de **señalar que algo salió mal** durante la ejecución. En lugar de que el programa simplemente se caiga, podemos *capturar* la excepción y decidir qué hacer.

### 5.1 Tipos comunes de excepción

| Excepción | Cuándo ocurre | Ejemplo |
|---|---|---|
| `ValueError` | Valor incorrecto pero de tipo correcto | `int("abc")` |
| `TypeError` | Operación con tipo inapropiado | `"2" + 3` |
| `FileNotFoundError` | Archivo o directorio no encontrado | `open("no_existe.txt")` |
| `KeyError` | Clave no existente en un dict | `{"a": 1}["b"]` |
| `ZeroDivisionError` | División por cero | `10 / 0` |
| `IndexError` | Índice fuera de rango | `[1,2][5]` |
| `AttributeError` | Atributo o método inexistente | `"hi".push("x")` |
| `StopIteration` | Agotador de iterable vacío | `next(iter([]))` |

In [ ]:
# ValueError — el valor no tiene sentido
try:
    numero = int("abc")
except ValueError as e:
    print(f"ValueError: {e}")

# TypeError — tipo de dato inapropiado
try:
    resultado = "hola" + 42
except TypeError as e:
    print(f"TypeError: {e}")

# KeyError — clave inexistente
try:
    mi_dict = {"nombre": "Ana"}
    valor = mi_dict["edad"]
except KeyError as e:
    print(f"KeyError: la clave {e} no existe")

# ZeroDivisionError
try:
    x = 10 / 0
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

### 5.2 Captura múltiple

In [ ]:
def dividir_seguro(a, b):
    """Divide a entre b con manejo de errores."""
    try:
        resultado = a / b
    except (TypeError, ZeroDivisionError) as e:
        # Captura ambos errores con un solo bloque
        print(f"  [ERROR] {type(e).__name__}: {e}")
        resultado = None
    return resultado

print("10 / 3:", dividir_seguro(10, 3))
print("10 / 0:", dividir_seguro(10, 0))
print("'10' / 3:", dividir_seguro("10", 3))
print("'10' / 0:", dividir_seguro("10", 0))

### 5.3 `else` y `finally`

In [ ]:
def cargar_configuracion(ruta):
    """Carga config con try/except/else/finally."""
    import json
    archivo = None
    try:
        archivo = open(ruta, "r", encoding="utf-8")
        config = json.load(archivo)
    except FileNotFoundError:
        print(f"  [except] Archivo '{ruta}' no encontrado. Usando defaults.")
        config = {"debug": False, "version": "1.0"}
    except json.JSONDecodeError as e:
        print(f"  [except] JSON corrupto: {e}")
        config = {}
    else:
        # Solo se ejecuta si NO hubo excepción
        print(f"  [else] Config cargada OK ({len(config)} claves)")
    finally:
        # SIEMPRE se ejecuta: cierra el archivo si se abrió
        if archivo:
            archivo.close()
            print("  [finally] Archivo cerrado.")
        else:
            print("  [finally] No había archivo que cerrar.")
    return config

print("--- Caso 1: archivo inexistente ---")
c1 = cargar_configuracion("no_existe.json")
print(f"Resultado: {c1}\n")

print("--- Caso 2: archivo válido ---")
from pathlib import Path
Path("ejemplo_archivos/config_ok.json").write_text(
    '{"debug": true, "timeout": 30}', encoding="utf-8"
)
c2 = cargar_configuracion("ejemplo_archivos/config_ok.json")
print(f"Resultado: {c2}")

### 5.4 `raise` — lanzar excepciones manualmente

In [ ]:
def calcular_descuento(precio, porcentaje):
    if not isinstance(precio, (int, float)) or precio < 0:
        raise TypeError(f"El precio debe ser un número positivo, recibido: {precio!r}")
    if not (0 < porcentaje <= 100):
        raise ValueError(f"El porcentaje debe estar entre 1 y 100, recibido: {porcentaje}")
    return precio * (1 - porcentaje / 100)

# Caso exitoso
try:
    resultado = calcular_descuento(200, 15)
    print(f"Descuento: ${200 - resultado:.2f}")
except (TypeError, ValueError) as e:
    print(f"Error: {e}")

# Caso con error
try:
    resultado = calcular_descuento(-50, 10)
except (TypeError, ValueError) as e:
    print(f"Error: {e}")

### 5.5 Excepciones propias

In [ ]:
class SaldoInsuficienteError(Exception):
    """Se lanza cuando el saldo es menor al monto solicitado."""
    def __init__(self, saldo, monto):
        self.saldo = saldo
        self.monto = monto
        super().__init__(
            f"Saldo insuficiente: tiene ${saldo:.2f}, necesita ${monto:.2f}"
        )


class CuentaBancaria:
    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self.saldo = saldo_inicial

    def retirar(self, monto):
        if monto <= 0:
            raise ValueError("El monto debe ser positivo")
        if monto > self.saldo:
            raise SaldoInsuficienteError(self.saldo, monto)
        self.saldo -= monto
        return self.saldo


cuenta = CuentaBancaria("Ana", 500)
print(f"Saldo inicial: ${cuenta.saldo:.2f}")

try:
    cuenta.retirar(200)
    print(f"Retiro OK. Saldo: ${cuenta.saldo:.2f}")
    cuenta.retirar(400)
except SaldoInsuficienteError as e:
    print(f"Error: {e}")
    print(f"  (Saldo actual: ${e.saldo:.2f}, necesitaba: ${e.monto:.2f})")

### 5.6 `assert` — validaciones en desarrollo

`assert` es útil para *precondiciones* y *depuración*. Si la condición es falsa, lanza `AssertionError`.

In [ ]:
def promedio_notas(notas):
    """Calcula el promedio de una lista de notas (0-100)."""
    assert isinstance(notas, list), "notas debe ser una lista"
    assert len(notas) > 0, "La lista no puede estar vacía"
    assert all(0 <= n <= 100 for n in notas), "Todas las notas deben estar entre 0 y 100"
    return sum(notas) / len(notas)

# Caso correcto
print(f"Promedio: {promedio_notas([85, 90, 78, 92]):.1f}")

# Caso con error (descomentar para ver el AssertionError)
# promedio_notas([])
# promedio_notas([10, 200, 50])

## 6. Jerarquía de excepciones

Todas las excepciones en Python heredan de `BaseException`. Las que usamos normalmente heredan de `Exception`.

In [ ]:
# Jerarquía simplificada:
#
# BaseException
# ├── KeyboardInterrupt    ← Ctrl+C
# ├── SystemExit           ← sys.exit()
# └── Exception            ← TODAS las que usamos normalmente
#     ├── ArithmeticError
#     │   ├── ZeroDivisionError
#     │   └── OverflowError
#     ├── LookupError
#     │   ├── IndexError
#     │   └── KeyError
#     ├── OSError
#     │   ├── FileNotFoundError
#     │   └── FileExistsError
#     ├── ValueError
#     ├── TypeError
#     └── AttributeError
#
# IMPORTANTE:
# - Capturar Exception captura todo lo de arriba.
# - Capturar Exception + ValueError NO captura TypeError (son hermanas).
# - NUNCA uses `except Exception` para "ocultar" errores.
# - Usa siempre tipos específicos cuando sea posible.

# Demostración práctica:
try:
    {1: "a"}[2]
except LookupError:  #父de KeyError
    print("Capturado como LookupError (clase padre de KeyError)")

try:
    int("abc")
except Exception as e:
    print(f"ValueError es subclase de Exception: {isinstance(e, Exception)}")
    print(f"Tipo real: {type(e).__name__}")
    print(f"MRO (Method Resolution Order):")
    for clase in type(e).__mro__:
        print(f"  → {clase.__name__}")

## Tabla de referencia

### Modos de apertura de archivos

| Modo | Nombre | Descripción |
|------|--------|-------------|
| `r` | Read | Lectura (por defecto). Falla si no existe. |
| `w` | Write | Escritura. **Sobrescribe** si ya existe, lo crea si no. |
| `a` | Append | Escritura al final. Crea si no existe. |
| `x` | Exclusive | Crea y escribe. **Falla** si ya existe. |
| `rb` | Read binary | Lectura de datos binarios. |
| `wb` | Write binary | Escritura de datos binarios. |
| `r+` | Read+Write | Lectura y escritura. El puntero empieza al inicio. |
| `w+` | Write+Read | Sobrescribe y permite leer. |
| `a+` | Append+Read | Escribe al final y permite leer. |

### Excepciones comunes

| Excepción | Cuándo | Ejemplo |
|---|---|---|
| `ValueError` | Valor inválido | `int("abc")` |
| `TypeError` | Tipo incorrecto | `len(42)` |
| `FileNotFoundError` | Archivo no existe | `open("x.txt")` |
| `FileExistsError` | Archivo ya existe (modo `x`) | `open("x.txt", "x")` |
| `KeyError` | Clave dict no existe | `d["missing"]` |
| `IndexError` | Índice fuera de rango | `[1,2][5]` |
| `ZeroDivisionError` | División por cero | `1/0` |
| `AttributeError` | Atributo inexistente | `"s".foo()` |
| `ImportError` | Módulo no encontrado | `import modulo_falso` |
| `StopIteration` | Iterable agotado | `next(iter([]))` |

## Ejercicios

### Ejercicio 1 (guiado) — Crear un inventario con pathlib y JSON

Completa el código para que:
1. Crea la carpeta `inventario` si no existe.
2. Escribe un archivo `productos.json` con una lista de al menos 4 productos.
3. Léealo de vuelta y muestra el producto más caro.

In [ ]:
import json
from pathlib import Path

# Paso 1: Crear carpeta
# COMPLETAR: crear carpeta "inventario" con mkdir

# Paso 2: Definir productos
productos = [
    # COMPLETAR: al menos 4 productos con nombre, precio y stock
]

# Paso 3: Guardar en JSON
# COMPLETAR: usar json.dump con indent=2

# Paso 4: Leer y encontrar el más caro
# COMPLETAR: usar json.load y la función max()
# print(f"Producto más caro: ...")

### Ejercicio 2 (guiado) — Procesar CSV con DictReader

Dado un archivo `notas.csv` con columnas `nombre, nota1, nota2, nota3`, escribe un script que:
1. Lea el archivo.
2. Calcule el promedio de cada estudiante.
3. Escriba un nuevo archivo `promedios.csv` con columnas `nombre, promedio`.

In [ ]:
import csv
from pathlib import Path

# Archivo de entrada
carpeta = Path("ejemplo_archivos")

# Datos de ejemplo (para que puedas probar)
datos = [
    ["nombre", "nota1", "nota2", "nota3"],
    ["Ana", 85, 90, 78],
    ["Carlos", 72, 68, 80],
    ["María", 95, 92, 88],
]
ruta_notas = carpeta / "notas.csv"
with open(ruta_notas, "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerows(datos)

# COMPLETAR:
# 1. Leer notas.csv con DictReader
# 2. Calcular promedio = (nota1 + nota2 + nota3) / 3
# 3. Escribir promedios.csv con DictWriter


### Ejercicio 3 (guiado) — Manejo de excepciones

Escribe una función `leer_numero(ruta)` que:
1. Lea un archivo de texto que contiene un número.
2. Si el archivo no existe → lance `FileNotFoundError` con un mensaje claro.
3. Si el contenido no es un número válido → lance `ValueError`.
4. Si todo está bien → retorne el número.

In [ ]:
from pathlib import Path

def leer_numero(ruta):
    """
    Lee un archivo y retorna su contenido como número.
    
    Raises:
        FileNotFoundError: si el archivo no existe.
        ValueError: si el contenido no es un número válido.
    """
    # COMPLETAR:
    # 1. Verificar que el archivo existe (usar Path.exists())
    #    Si no existe, raise FileNotFoundError(...)
    # 2. Leer el contenido con .read_text()
    # 3. Intentar convertir a float (usar try/except ValueError)
    # 4. Retornar el número
    pass


# Pruebas
# Crear archivo de prueba
Path("ejemplo_archivos/numero.txt").write_text("3.14", encoding="utf-8")

# Descomentar las pruebas:
# print(leer_numero("ejemplo_archivos/numero.txt"))    # → 3.14
# print(leer_numero("ejemplo_archivos/no_existe.txt")) # → FileNotFoundError

# Crear archivo con texto inválido
Path("ejemplo_archivos/numero_mal.txt").write_text("hola", encoding="utf-8")
# print(leer_numero("ejemplo_archivos/numero_mal.txt")) # → ValueError

### Ejercicio 4 (independiente) — Línea de comandos de archivos

Crea una función `explorar_carpeta(ruta_str)` que:

1. Reciba una ruta como string.
2. Use `pathlib` para explorarla.
3. Agrupe los archivos por extensión.
4. Retorne un diccionario `{".py": 5, ".csv": 2, ".json": 1}`.
5. Si la ruta no existe, lance un `FileNotFoundError` con un mensaje descriptivo.

**Pistas:** Usa `Path.suffix` para obtener la extensión y un diccionario para contar.

In [ ]:
from pathlib import Path

def explorar_carpeta(ruta_str):
    """
    Explora una carpeta y cuenta archivos por extensión.
    
    Args:
        ruta_str: Ruta de la carpeta como string.
    
    Returns:
        dict: {extensión: cantidad} de archivos encontrados.
    
    Raises:
        FileNotFoundError: si la ruta no existe o no es un directorio.
    """
    # COMPLETAR
    pass


# Prueba con una carpeta real del sistema
# print(explorar_carpeta("C:/Users/luisj/Documents"))

## Resumen

| Tema | Concepto clave | Ejemplo |
|------|----------------|---------|
| **pathlib** | Rutas como objetos, portable entre SO | `Path("datos") / "archivo.txt"` |
| **pathlib ops** | Crear, verificar, listar, leer, escribir | `.mkdir()`, `.exists()`, `.glob()`, `.read_text()` |
| **with** | Cierra archivos automáticamente | `with open(...) as f:` |
| **Modos** | r lectura, w sobrescribe, a agrega, x exclusivo | `open("f.txt", "w")` |
| **json.dump** | Python → archivo JSON | `json.dump(data, f, indent=2)` |
| **json.load** | Archivo JSON → Python | `json.load(f)` |
| **json.dumps** | Python → string JSON | `json.dumps(data)` |
| **csv.reader** | Lee CSV como listas | `csv.reader(f)` |
| **csv.DictReader** | Lee CSV como diccionarios | `csv.DictReader(f)` |
| **csv.writer** | Escribe listas como CSV | `csv.writer(f).writerows(data)` |
| **csv.DictWriter** | Escribe diccionarios como CSV | `csv.DictWriter(f, fieldnames=[...])` |
| **try/except** | Captura errores específicos | `except ValueError as e:` |
| **else** | Se ejecuta si NO hubo error | `else: print("OK")` |
| **finally** | SIEMPRE se ejecuta | `finally: f.close()` |
| **raise** | Lanza excepción manualmente | `raise ValueError("msg")` |
| **Excepciones propias** | Clase que hereda de `Exception` | `class MiError(Exception):` |
| **assert** | Validación para desarrollo | `assert x > 0, "x debe ser positivo"` |
| **Jerarquía** | BaseException → Exception → tipos específicos | Herencia de excepciones |